# Spatial Intelligence - Part 2

## 1. Import the needed libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color

## 2. Check the TopologicPy Version

In [2]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.33) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [3]:
renderer = "vscode"

## 4. Utility functions to reset the face dictionaries and transfer dictionaries by key

In [4]:
def reset_dictionaries(shell):
    faces = Topology.Faces(shell)
    for i, f in enumerate(faces):
        d = Topology.Dictionary(f)
        keys = Dictionary.Keys(d)
        for key in keys:
            if not key == "face_id":
                d = Dictionary.RemoveKey(d, key)
        f = Topology.SetDictionary(f, d)

def transfer_dicts_by_key(topologies, selectors, key):
    dicts = {}
    for t in topologies:
        d = Topology.Dictionary(t)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            dicts[str(value)] = t
    
    for s in selectors:
        d = Topology.Dictionary(s)
        value = Dictionary.ValueAtKey(d, key, None)
        if value:
            f = dicts.get(str(value), None)
            if f:
                f = Topology.SetDictionary(f, d)


## 5. Import the gallery floor plan

In [5]:
gallery = Topology.ByBREPPath(r"C:\IaaC\AIA\GraphML\Assignments\Assignment02\Assets\MusashinoLibrary_FloorPlate.brep")

## 6. Show the geometry

In [6]:
Topology.Show(gallery,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

## 7. Create a grid overlay

In [7]:
b_r = Wire.BoundingRectangle(gallery)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")
uRange = list(range(0,int(width)+3,3))
vRange = list(range(0,int(length)+3,3))

grid = Grid.EdgesByDistances(gallery, clip=True, uRange=uRange, vRange=vRange)

## 8. Show the geometry and the grid

In [8]:
Topology.Show(gallery, grid,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="grey",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,height=600,
              renderer = renderer)

## 9. Slice the floor plan with the grid to create a topologic shell

In [9]:
shell = Topology.Slice(gallery, grid)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

## 10. Derive analysis graphs from the shell

In [10]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

## 11. Derive and store the graph vertices

In [11]:
g_verts = Graph.Vertices(analysis_graph)

## 12. Spatial Intelligence through Graph Analysis

### a. Community Detection (About 5 minutes)

In [12]:
community_list = Graph.CommunityPartition(analysis_graph, colorScale="thermal")

In [13]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [14]:
Topology.Show(faces,
              faceColorKey="cp_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

### b. Degree centrality

Bin By Dictionary Key
* Use the community partition number (or colour) to separate the faces of the shell into different bins or categories
* Derive the outer boundary of each face group (perimeter) and make a face out of that perimeter

In [15]:
bins = Topology.BinByDictionaryKey(faces, key="community")
bin_dict = bins[0]
keys = list(bin_dict.keys())
face_groups = []
for key in keys:
    bin_faces = bin_dict[key]
    temp_shell = Shell.ByFaces(bin_faces)
    eb = Shell.ExternalBoundary(temp_shell)
    eb = Wire.RemoveCollinearEdges(eb)
    eb = Face.ByWire(eb)
    face_groups.append(eb)



Show the result

In [16]:
Topology.Show(face_groups,
              faceOpacity=1,
              showEdges=True,
              edgeWidth=8,
              edgeColor="grey",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new shell from the new faces

In [17]:
new_shell = Shell.ByFaces(face_groups)


Show the result

In [18]:
Topology.Show(new_shell,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Create a new graph from the new shell

In [19]:
new_graph = Graph.ByTopology(new_shell)
new_verts = Graph.Vertices(new_graph)
for v in new_verts:
    d = Dictionary.ByKeysValues(["color", "size"], ["red", 12])
    v = Topology.SetDictionary(v, d)

Show the result

In [20]:
Topology.Show(new_shell, new_graph,
              faceOpacity=0.9,
              showEdges=True,
              showVertices=True,
              vertexSizeKey="size",
              vertexColorKey="color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

Compute degree centralities

In [21]:
degree_centralities = Graph.DegreeCentrality(new_graph, normalize=False)

Transfer/Interpolate values from the new graph vertices to the original graph vertices

In [22]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=3, key="degree_centrality")

Derive the colour of each vertex based on the interpolated value

In [23]:
minValue = min(degree_centralities)
maxValue = max(degree_centralities)
for v in g_verts:
    d = Topology.Dictionary(v)
    d_c = Dictionary.ValueAtKey(d, "degree_centrality")
    color = Color.AnyToHex(Color.ByValueInRange(d_c, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "dc_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

Transfer the information from the graph vertices to the faces of the original shell

In [24]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

Show the result

In [25]:
Topology.Show(faces,
              faceColorKey="dc_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              vertexSizeKey="size",
              vertexColorKey="dc_color",
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)